# 13. M1 Expanded Training Fixed-Eval 검증

목표는 모델/feature/threshold를 바꾸지 않고, 학습 데이터만 확장했을 때 기존 `strict_no_event20` 49행 평가 기준 성능이 유지되는지 확인하는 것이다.

고정값:
- fixed eval: `strict_no_event20` 49행
- feature: `compact13_overlap`
- model: `LogisticRegression(class_weight="balanced")`
- threshold: `0.6`
- group: `substation_id`


In [1]:
from __future__ import annotations

from functools import lru_cache
from pathlib import Path
import re
import zipfile
import warnings

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

DATA_DIR = Path("05_데이터셋") / "PreDist"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
NOTEBOOK_PATH = Path("06_노트북") / "13_m1_expanded_training_fixed_eval_validation.ipynb"
OUTPUT_DIR = Path("07_데이터산출물")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = OUTPUT_DIR / "m1_feature_expansion_features.csv"
FEATURE_SET_SUMMARY_PATH = OUTPUT_DIR / "m1_compact_feature_set_summary.csv"
HARD_NORMAL_REVIEW_PATH = OUTPUT_DIR / "m1_hard_normal_timeseries_review.csv"

CANDIDATE_AUDIT_PATH = OUTPUT_DIR / "m1_expansion_candidate_audit.csv"
FEATURE_POOL_PATH = OUTPUT_DIR / "m1_expansion_feature_pool.csv"
CV_METRICS_PATH = OUTPUT_DIR / "m1_expanded_training_fixed_eval_cv_metrics.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "m1_expanded_training_fixed_eval_predictions.csv"
DECISION_MATRIX_PATH = OUTPUT_DIR / "m1_expanded_training_fixed_eval_decision_matrix.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "m1_expanded_training_fixed_eval_feature_importance.csv"
REPORT_PATH = OUTPUT_DIR / "13_M1_expanded_training_fixed_eval_보고서.md"

SOURCE_PREFIX = "manufacturer 1"
THRESHOLD = 0.6
EXPECTED_7D_SAMPLE_COUNT = 7 * 24 * 6
EXCLUDED_EVENT_IDS = {20, 34, 69}
RANDOM_STATE = 42


## 1. 입력 로드와 fixed eval 확인


In [2]:
features = pd.read_csv(FEATURES_PATH)
feature_set_summary = pd.read_csv(FEATURE_SET_SUMMARY_PATH)
hard_normal_review = pd.read_csv(HARD_NORMAL_REVIEW_PATH)

for frame in [features, hard_normal_review]:
    if "source_event_id" in frame.columns:
        frame["source_event_id"] = frame["source_event_id"].astype(int)
    if "substation_id" in frame.columns:
        frame["substation_id"] = frame["substation_id"].astype(int)
    for col in ["window_start", "window_end"]:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col])

compact_row = feature_set_summary.loc[
    feature_set_summary["feature_set"].eq("compact13_overlap")
].iloc[0]
COMPACT13_FEATURES = compact_row["features"].split("|")

fixed_eval = features.copy()
fixed_eval["source_event_id"] = fixed_eval["source_event_id"].astype(int)
fixed_eval["substation_id"] = fixed_eval["substation_id"].astype(int)
fixed_eval["window_start"] = pd.to_datetime(fixed_eval["window_start"])
fixed_eval["window_end"] = pd.to_datetime(fixed_eval["window_end"])
fixed_eval["y"] = fixed_eval["label"].map({"normal": 0, "efd_possible": 1}).astype(int)
fixed_eval["pool_role"] = "fixed_eval"
fixed_eval["candidate_type"] = "fixed_eval"
fixed_eval["review_tag"] = ""

review_required_events = set(
    hard_normal_review.loc[
        hard_normal_review["review_label"].eq("review_required_normal"), "source_event_id"
    ].astype(int)
)
hard_normal_events = set(hard_normal_review["source_event_id"].astype(int))
fixed_eval.loc[
    fixed_eval["source_event_id"].isin(review_required_events), "review_tag"
] = "review_required_normal"
fixed_eval.loc[
    fixed_eval["source_event_id"].isin(hard_normal_events - review_required_events), "review_tag"
] = "hard_normal_metadata"

assert len(COMPACT13_FEATURES) == 13
assert all(feature in fixed_eval.columns for feature in COMPACT13_FEATURES)
assert len(fixed_eval) == 49
assert fixed_eval["label"].value_counts().to_dict() == {"normal": 35, "efd_possible": 14}
assert not fixed_eval["source_event_id"].isin(EXCLUDED_EVENT_IDS).any()
assert review_required_events == {19, 68}

fixed_eval[["sample_id", "source_event_id", "substation_id", "label", "review_tag"]].head()


,sample_id,source_event_id,substation_id,label,review_tag
0,normal_0002,2,9,normal,
1,normal_0004,4,15,normal,
2,normal_0008,8,6,normal,hard_normal_metadata
3,normal_0012,12,19,normal,
4,normal_0014,14,26,normal,


## 2. 원본 데이터 로더와 feature 계산 함수


In [3]:
def read_m1_csv(name: str, **kwargs) -> pd.DataFrame:
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(f"{SOURCE_PREFIX}/{name}") as handle:
            return pd.read_csv(handle, sep=";", **kwargs)


faults = read_m1_csv("faults.csv")
disturbances = read_m1_csv("disturbances.csv")

for col in [
    "Report date",
    "Possible anomaly start",
    "Possible anomaly end",
    "Training start",
    "Training end",
]:
    faults[col] = pd.to_datetime(faults[col], errors="coerce")
disturbances["Event start"] = pd.to_datetime(disturbances["Event start"], errors="coerce")
faults["Event ID"] = faults["Event ID"].astype(int)
faults["substation ID"] = faults["substation ID"].astype(int)
disturbances["substation ID"] = disturbances["substation ID"].astype(int)


@lru_cache(maxsize=6)
def load_operational(substation_id: int) -> pd.DataFrame:
    df = read_m1_csv(f"operational_data/substation_{int(substation_id)}.csv")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    for col in df.columns:
        if col != "timestamp":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["s_hc1_supply_temperature_error"] = (
        df["s_hc1_supply_temperature"] - df["s_hc1_supply_temperature_setpoint"]
    )
    df["p_return_gap"] = df["p_hc1_return_temperature"] - df["p_net_return_temperature"]
    return df.sort_values("timestamp").reset_index(drop=True)


def window_bounds(df: pd.DataFrame, start, end) -> tuple[int, int]:
    ts = df["timestamp"].to_numpy(dtype="datetime64[ns]")
    start64 = np.datetime64(pd.Timestamp(start).to_datetime64(), "ns")
    end64 = np.datetime64(pd.Timestamp(end).to_datetime64(), "ns")
    left = int(np.searchsorted(ts, start64, side="left"))
    right = int(np.searchsorted(ts, end64, side="left"))
    return left, right


def window_slice(df: pd.DataFrame, start, end) -> pd.DataFrame:
    left, right = window_bounds(df, start, end)
    return df.iloc[left:right].copy()


def sample_count_for_window(substation_id: int, start, end) -> int:
    df = load_operational(int(substation_id))
    left, right = window_bounds(df, start, end)
    return right - left


def last_minus_first(values: pd.Series) -> float:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    if len(clean) < 2:
        return np.nan
    return float(clean.iloc[-1] - clean.iloc[0])


def period_stat(window: pd.DataFrame, signal: str, start, end, stat: str) -> float:
    subset = window.loc[
        (window["timestamp"] >= pd.Timestamp(start))
        & (window["timestamp"] < pd.Timestamp(end)),
        signal,
    ]
    if subset.dropna().empty:
        return np.nan
    if stat == "mean":
        return float(subset.mean())
    if stat == "std":
        return float(subset.std(ddof=0))
    raise ValueError(stat)


def compute_feature(window: pd.DataFrame, signal: str, feature_stat: str, window_start, window_end) -> float:
    if signal not in window.columns:
        return np.nan
    window_start = pd.Timestamp(window_start)
    window_end = pd.Timestamp(window_end)
    if feature_stat == "last_minus_first":
        return last_minus_first(window[signal])
    if feature_stat == "last_1d_mean_minus_prev_6d_mean":
        return period_stat(window, signal, window_end - pd.Timedelta(days=1), window_end, "mean") - period_stat(
            window, signal, window_start, window_end - pd.Timedelta(days=1), "mean"
        )
    if feature_stat == "last_12h_mean_minus_prev_12h_mean":
        return period_stat(window, signal, window_end - pd.Timedelta(hours=12), window_end, "mean") - period_stat(
            window, signal, window_end - pd.Timedelta(hours=24), window_end - pd.Timedelta(hours=12), "mean"
        )
    if feature_stat == "last_6h_mean_minus_prev_6h_mean":
        return period_stat(window, signal, window_end - pd.Timedelta(hours=6), window_end, "mean") - period_stat(
            window, signal, window_end - pd.Timedelta(hours=12), window_end - pd.Timedelta(hours=6), "mean"
        )
    if feature_stat == "last_1d_std_minus_prev_6d_std":
        return period_stat(window, signal, window_end - pd.Timedelta(days=1), window_end, "std") - period_stat(
            window, signal, window_start, window_end - pd.Timedelta(days=1), "std"
        )
    raise ValueError(feature_stat)


def compute_compact13_features(substation_id: int, window_start, window_end) -> tuple[dict, int, float]:
    df = load_operational(int(substation_id))
    window = window_slice(df, window_start, window_end)
    row = {}
    for feature in COMPACT13_FEATURES:
        signal, feature_stat = feature.split("__", 1)
        row[feature] = compute_feature(window, signal, feature_stat, window_start, window_end)
    sample_count = int(len(window))
    coverage_rate = sample_count / EXPECTED_7D_SAMPLE_COUNT
    return row, sample_count, coverage_rate


## 3. weak positive 후보 생성


In [4]:
fixed_positive_ids = set(
    fixed_eval.loc[fixed_eval["label"].eq("efd_possible"), "source_event_id"].astype(int)
)

candidate_audit_rows = []
candidate_feature_rows = []

for _, fault in faults.sort_values("Event ID").iterrows():
    event_id = int(fault["Event ID"])
    substation_id = int(fault["substation ID"])
    report_date = fault["Report date"]
    efd_possible = bool(fault["efd_possible"])
    fault_label = fault["Fault label"]
    window_start = pd.NaT
    window_end = pd.NaT
    sample_count = 0
    coverage_rate = np.nan
    audit_status = "excluded"
    exclusion_reason = ""

    if not efd_possible:
        exclusion_reason = "efd_possible_false"
    elif pd.isna(report_date):
        exclusion_reason = "missing_report_date"
    elif event_id in EXCLUDED_EVENT_IDS:
        exclusion_reason = "excluded_event_policy"
    elif event_id in fixed_positive_ids:
        exclusion_reason = "duplicate_fixed_eval"
    else:
        window_end = pd.Timestamp(report_date)
        window_start = window_end - pd.Timedelta(days=7)
        feature_values, sample_count, coverage_rate = compute_compact13_features(
            substation_id, window_start, window_end
        )
        if coverage_rate < 0.95:
            exclusion_reason = "low_coverage"
        else:
            audit_status = "accepted"
            exclusion_reason = ""
            candidate_feature_rows.append(
                {
                    "sample_id": f"weak_positive_{event_id:04d}",
                    "pool_role": "expansion_candidate",
                    "candidate_type": "weak_positive",
                    "manufacturer": "manufacturer_1",
                    "label": "efd_possible",
                    "label_strength": "weak_positive",
                    "y": 1,
                    "source_event_id": event_id,
                    "substation_id": substation_id,
                    "window_start": window_start,
                    "window_end": window_end,
                    "window_days": 7.0,
                    "window_policy": "report_pre_7d",
                    "sample_count": sample_count,
                    "expected_sample_count": EXPECTED_7D_SAMPLE_COUNT,
                    "coverage_rate": coverage_rate,
                    "source_type": "fault_record",
                    "review_tag": "",
                    "fault_label": fault_label,
                    **feature_values,
                }
            )

    candidate_audit_rows.append(
        {
            "candidate_kind": "weak_positive",
            "candidate_id": f"weak_positive_{event_id:04d}",
            "source_event_id": event_id,
            "substation_id": substation_id,
            "window_start": window_start,
            "window_end": window_end,
            "sample_count": sample_count,
            "expected_sample_count": EXPECTED_7D_SAMPLE_COUNT,
            "coverage_rate": coverage_rate,
            "audit_status": audit_status,
            "exclusion_reason": exclusion_reason,
            "fault_label": fault_label,
            "efd_possible": efd_possible,
        }
    )

positive_candidate_features = pd.DataFrame(candidate_feature_rows)
positive_candidate_features[["sample_id", "source_event_id", "substation_id", "coverage_rate"]]


,sample_id,source_event_id,substation_id,coverage_rate
0,weak_positive_0005,5,11,1.0
1,weak_positive_0006,6,21,1.0
2,weak_positive_0013,13,24,1.0
3,weak_positive_0023,23,27,1.0
4,weak_positive_0024,24,13,1.0
5,weak_positive_0029,29,17,1.0
6,weak_positive_0032,32,21,1.0
7,weak_positive_0037,37,19,1.0
8,weak_positive_0038,38,24,1.0
9,weak_positive_0045,45,31,1.0


## 4. disturbance/fault-free candidate normal 생성


In [5]:
def build_fault_windows() -> pd.DataFrame:
    rows = []
    for _, fault in faults.dropna(subset=["Report date"]).iterrows():
        event_id = int(fault["Event ID"])
        substation_id = int(fault["substation ID"])
        end = pd.Timestamp(fault["Report date"])
        rows.append(
            {
                "source_event_id": event_id,
                "substation_id": substation_id,
                "window_start": end - pd.Timedelta(days=7),
                "window_end": end,
            }
        )
    return pd.DataFrame(rows)


fault_windows = build_fault_windows()
eval_windows = fixed_eval[["source_event_id", "substation_id", "window_start", "window_end"]].copy()


def overlaps_window(windows: pd.DataFrame, substation_id: int, start, end) -> bool:
    same = windows.loc[windows["substation_id"].eq(int(substation_id))]
    if same.empty:
        return False
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    return bool(((same["window_start"] < end) & (same["window_end"] > start)).any())


def disturbance_count_in_window(substation_id: int, start, end) -> int:
    same = disturbances.loc[disturbances["substation ID"].eq(int(substation_id))]
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    return int(((same["Event start"] >= start) & (same["Event start"] < end)).sum())


def operational_substation_ids() -> list[int]:
    with zipfile.ZipFile(ZIP_PATH) as zf:
        ids = []
        for name in zf.namelist():
            match = re.match(rf"{re.escape(SOURCE_PREFIX)}/operational_data/substation_(\d+)\.csv", name)
            if match:
                ids.append(int(match.group(1)))
        return sorted(ids)


normal_candidate_rows = []
normal_feature_rows = []
accepted_normal_starts: dict[int, list[pd.Timestamp]] = {}

for substation_id in operational_substation_ids():
    df = load_operational(substation_id)
    start_cursor = df["timestamp"].min().ceil("D")
    max_start = df["timestamp"].max() - pd.Timedelta(days=7)
    accepted_normal_starts[substation_id] = []
    candidate_index = 0

    while start_cursor <= max_start:
        candidate_index += 1
        window_start = pd.Timestamp(start_cursor)
        window_end = window_start + pd.Timedelta(days=7)
        sample_count = sample_count_for_window(substation_id, window_start, window_end)
        coverage_rate = sample_count / EXPECTED_7D_SAMPLE_COUNT
        disturbance_count = disturbance_count_in_window(substation_id, window_start, window_end)
        overlap_fault = overlaps_window(fault_windows, substation_id, window_start, window_end)
        overlap_eval = overlaps_window(eval_windows, substation_id, window_start, window_end)
        far_enough = all(
            abs((window_start - prev).days) >= 30
            for prev in accepted_normal_starts[substation_id]
        )
        under_quota = len(accepted_normal_starts[substation_id]) < 2

        exclusion_reasons = []
        if coverage_rate < 0.95:
            exclusion_reasons.append("low_coverage")
        if disturbance_count > 0:
            exclusion_reasons.append("disturbance_overlap")
        if overlap_fault:
            exclusion_reasons.append("fault_window_overlap")
        if overlap_eval:
            exclusion_reasons.append("fixed_eval_window_overlap")
        if not far_enough:
            exclusion_reasons.append("selected_window_too_close")
        if not under_quota:
            exclusion_reasons.append("substation_quota_full")

        audit_status = "accepted" if not exclusion_reasons else "excluded"
        candidate_id = f"candidate_normal_s{substation_id:02d}_{window_start:%Y%m%d}"
        normal_candidate_rows.append(
            {
                "candidate_kind": "candidate_normal",
                "candidate_id": candidate_id,
                "source_event_id": np.nan,
                "substation_id": substation_id,
                "window_start": window_start,
                "window_end": window_end,
                "sample_count": sample_count,
                "expected_sample_count": EXPECTED_7D_SAMPLE_COUNT,
                "coverage_rate": coverage_rate,
                "disturbance_count": disturbance_count,
                "fault_window_overlap": overlap_fault,
                "fixed_eval_window_overlap": overlap_eval,
                "audit_status": audit_status,
                "exclusion_reason": "|".join(exclusion_reasons),
            }
        )

        if audit_status == "accepted":
            feature_values, _, _ = compute_compact13_features(substation_id, window_start, window_end)
            accepted_normal_starts[substation_id].append(window_start)
            normal_feature_rows.append(
                {
                    "sample_id": candidate_id,
                    "pool_role": "expansion_candidate",
                    "candidate_type": "candidate_normal",
                    "manufacturer": "manufacturer_1",
                    "label": "candidate_normal",
                    "label_strength": "candidate_normal",
                    "y": 0,
                    "source_event_id": np.nan,
                    "substation_id": substation_id,
                    "window_start": window_start,
                    "window_end": window_end,
                    "window_days": 7.0,
                    "window_policy": "candidate_normal_7d",
                    "sample_count": sample_count,
                    "expected_sample_count": EXPECTED_7D_SAMPLE_COUNT,
                    "coverage_rate": coverage_rate,
                    "source_type": "candidate_normal_window",
                    "review_tag": "",
                    "fault_label": "",
                    **feature_values,
                }
            )

        start_cursor += pd.Timedelta(days=30)

normal_candidate_features = pd.DataFrame(normal_feature_rows)
normal_candidate_features[["sample_id", "substation_id", "window_start", "coverage_rate"]].head()


,sample_id,substation_id,window_start,coverage_rate
0,candidate_normal_s01_20180611,1,2018-06-11,1.000000
1,candidate_normal_s01_20180711,1,2018-07-11,1.000000
2,candidate_normal_s02_20150216,2,2015-02-16,1.000000
3,candidate_normal_s02_20150318,2,2015-03-18,1.000000
4,candidate_normal_s03_20151110,3,2015-11-10,0.992063


## 5. feature pool 저장


In [6]:
candidate_audit = pd.concat(
    [pd.DataFrame(candidate_audit_rows), pd.DataFrame(normal_candidate_rows)],
    ignore_index=True,
    sort=False,
)
candidate_audit.to_csv(CANDIDATE_AUDIT_PATH, index=False, encoding="utf-8-sig")

fixed_pool_cols = [
    "sample_id",
    "pool_role",
    "candidate_type",
    "manufacturer",
    "label",
    "label_strength",
    "y",
    "source_event_id",
    "substation_id",
    "window_start",
    "window_end",
    "window_days",
    "window_policy",
    "sample_count",
    "expected_sample_count",
    "coverage_rate",
    "source_type",
    "review_tag",
    "fault_label",
] + COMPACT13_FEATURES

fixed_pool = fixed_eval.copy()
fixed_pool["manufacturer"] = "manufacturer_1"
fixed_pool["fault_label"] = fixed_pool["fault_label"].fillna("")
fixed_pool = fixed_pool[fixed_pool_cols]

feature_pool = pd.concat(
    [fixed_pool, positive_candidate_features, normal_candidate_features],
    ignore_index=True,
    sort=False,
)
feature_pool.to_csv(FEATURE_POOL_PATH, index=False, encoding="utf-8-sig")

pool_summary = feature_pool.groupby(["pool_role", "candidate_type", "label", "y"]).size().reset_index(name="rows")
pool_summary


,pool_role,candidate_type,label,y,rows
0,expansion_candidate,candidate_normal,candidate_normal,0,70
1,expansion_candidate,weak_positive,efd_possible,1,12
2,fixed_eval,fixed_eval,efd_possible,1,14
3,fixed_eval,fixed_eval,normal,0,35


## 6. leakage-safe fixed eval 학습/평가


In [7]:
def make_logistic_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                    max_iter=1000,
                ),
            ),
        ]
    )


def class_one_probability(model, X) -> np.ndarray:
    probabilities = model.predict_proba(X)
    if 1 not in model.classes_:
        return np.zeros(len(X))
    class_index = list(model.classes_).index(1)
    return probabilities[:, class_index]


fixed_eval_model_rows = feature_pool.loc[feature_pool["pool_role"].eq("fixed_eval")].copy()
candidate_model_rows = feature_pool.loc[feature_pool["pool_role"].eq("expansion_candidate")].copy()

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pred_rows = []
fold_audit_rows = []

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(
        fixed_eval_model_rows[COMPACT13_FEATURES],
        fixed_eval_model_rows["y"],
        fixed_eval_model_rows["substation_id"],
    ),
    start=1,
):
    fixed_train = fixed_eval_model_rows.iloc[train_idx].copy()
    fixed_test = fixed_eval_model_rows.iloc[test_idx].copy()
    test_groups = set(fixed_test["substation_id"].astype(int))
    train_groups = set(fixed_train["substation_id"].astype(int))
    assert train_groups.isdisjoint(test_groups)

    candidate_train = candidate_model_rows.loc[
        ~candidate_model_rows["substation_id"].astype(int).isin(test_groups)
    ].copy()
    assert set(candidate_train["substation_id"].astype(int)).isdisjoint(test_groups)

    strategies = {
        "dummy_most_frequent": (DummyClassifier(strategy="most_frequent"), fixed_train),
        "reference_compact13": (make_logistic_pipeline(), fixed_train),
        "expanded_compact13": (
            make_logistic_pipeline(),
            pd.concat([fixed_train, candidate_train], ignore_index=True),
        ),
    }

    fold_audit_rows.append(
        {
            "fold": fold,
            "test_rows": len(fixed_test),
            "test_groups": "|".join(str(g) for g in sorted(test_groups)),
            "fixed_train_rows": len(fixed_train),
            "candidate_train_rows": len(candidate_train),
            "candidate_train_positive": int(candidate_train["y"].eq(1).sum()),
            "candidate_train_normal": int(candidate_train["y"].eq(0).sum()),
            "group_overlap_count": len(train_groups.intersection(test_groups)),
        }
    )

    for strategy_name, (model, train_rows) in strategies.items():
        X_train = train_rows[COMPACT13_FEATURES]
        y_train = train_rows["y"].astype(int)
        X_test = fixed_test[COMPACT13_FEATURES]
        model.fit(X_train, y_train)
        y_probability = class_one_probability(model, X_test)
        y_pred = (y_probability >= THRESHOLD).astype(int)

        for row, probability, pred in zip(fixed_test.itertuples(index=False), y_probability, y_pred):
            pred_rows.append(
                {
                    "dataset_id": "strict_no_event20_fixed_eval",
                    "strategy": strategy_name,
                    "fold": fold,
                    "threshold": THRESHOLD,
                    "sample_id": row.sample_id,
                    "source_event_id": int(row.source_event_id),
                    "substation_id": int(row.substation_id),
                    "label": row.label,
                    "review_tag": row.review_tag,
                    "y_true": int(row.y),
                    "y_probability": float(probability),
                    "y_pred": int(pred),
                    "prediction_label": "efd_possible" if int(pred) == 1 else "normal",
                    "coverage_rate": float(row.coverage_rate),
                    "train_fixed_rows": int(len(fixed_train)),
                    "train_candidate_rows": int(len(candidate_train))
                    if strategy_name == "expanded_compact13"
                    else 0,
                    "train_candidate_positive": int(candidate_train["y"].eq(1).sum())
                    if strategy_name == "expanded_compact13"
                    else 0,
                    "train_candidate_normal": int(candidate_train["y"].eq(0).sum())
                    if strategy_name == "expanded_compact13"
                    else 0,
                }
            )

predictions = pd.DataFrame(pred_rows)
fold_audit = pd.DataFrame(fold_audit_rows)
predictions.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

fold_audit


,fold,test_rows,test_groups,fixed_train_rows,candidate_train_rows,candidate_train_positive,candidate_train_normal,group_overlap_count
0,1,11,8|10|11|13|22|25,38,67,9,58,0
1,2,9,2|6|15|28|30,40,72,12,60,0
2,3,11,3|7|9|17|18|20,38,69,11,58,0
3,4,9,1|4|16|21|26|29,40,67,9,58,0
4,5,9,5|12|14|19|23|24,40,67,9,58,0


## 7. metrics와 decision matrix


In [8]:
def metric_row(data: pd.DataFrame, strategy: str, evaluation_scope: str, fold="all") -> dict:
    y_true = data["y_true"].astype(int)
    y_pred = data["y_pred"].astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "strategy": strategy,
        "evaluation_scope": evaluation_scope,
        "fold": fold,
        "n": int(len(data)),
        "normal_n": int((data["y_true"] == 0).sum()),
        "positive_n": int((data["y_true"] == 1).sum()),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "false_positive_count": int(fp),
        "false_negative_count": int(fn),
        "true_positive_count": int(tp),
        "true_negative_count": int(tn),
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else 0.0,
        "hard_normal_35_48_fp_count": int(
            data.loc[data["source_event_id"].isin([35, 48]), "y_pred"].sum()
        ),
        "review_required_19_68_fp_count": int(
            data.loc[data["source_event_id"].isin([19, 68]), "y_pred"].sum()
        ),
    }


metric_rows = []
for strategy, strategy_data in predictions.groupby("strategy"):
    scopes = {
        "main_all_49": strategy_data,
        "sensitivity_exclude_review_required_19_68": strategy_data.loc[
            ~(
                strategy_data["label"].eq("normal")
                & strategy_data["source_event_id"].isin(review_required_events)
            )
        ],
    }
    for scope_name, scope_data in scopes.items():
        metric_rows.append(metric_row(scope_data, strategy, scope_name, fold="all"))
        for fold, fold_data in scope_data.groupby("fold"):
            metric_rows.append(metric_row(fold_data, strategy, scope_name, fold=int(fold)))

cv_metrics = pd.DataFrame(metric_rows).sort_values(["evaluation_scope", "strategy", "fold"])
cv_metrics.to_csv(CV_METRICS_PATH, index=False, encoding="utf-8-sig")


def aggregate_metric(strategy: str, scope: str) -> pd.Series:
    row = cv_metrics.loc[
        cv_metrics["strategy"].eq(strategy)
        & cv_metrics["evaluation_scope"].eq(scope)
        & cv_metrics["fold"].astype(str).eq("all")
    ]
    assert len(row) == 1
    return row.iloc[0]


ref_main = aggregate_metric("reference_compact13", "main_all_49")
exp_main = aggregate_metric("expanded_compact13", "main_all_49")
ref_sens = aggregate_metric("reference_compact13", "sensitivity_exclude_review_required_19_68")
exp_sens = aggregate_metric("expanded_compact13", "sensitivity_exclude_review_required_19_68")

ba_delta = float(exp_main["balanced_accuracy"] - ref_main["balanced_accuracy"])
recall_delta = float(exp_main["recall"] - ref_main["recall"])
fpr_delta = float(exp_main["false_positive_rate"] - ref_main["false_positive_rate"])
sensitivity_ba_delta = float(exp_sens["balanced_accuracy"] - ref_sens["balanced_accuracy"])
sensitivity_fpr_delta = float(exp_sens["false_positive_rate"] - ref_sens["false_positive_rate"])
hard_normal_fp_delta = int(exp_main["hard_normal_35_48_fp_count"] - ref_main["hard_normal_35_48_fp_count"])

criteria = [
    {
        "criterion": "balanced_accuracy_not_drop_over_0_02",
        "reference_value": float(ref_main["balanced_accuracy"]),
        "expanded_value": float(exp_main["balanced_accuracy"]),
        "delta": ba_delta,
        "pass": ba_delta >= -0.02,
    },
    {
        "criterion": "recall_not_drop_over_0_05",
        "reference_value": float(ref_main["recall"]),
        "expanded_value": float(exp_main["recall"]),
        "delta": recall_delta,
        "pass": recall_delta >= -0.05,
    },
    {
        "criterion": "fpr_not_worse_over_0_05",
        "reference_value": float(ref_main["false_positive_rate"]),
        "expanded_value": float(exp_main["false_positive_rate"]),
        "delta": fpr_delta,
        "pass": fpr_delta <= 0.05,
    },
    {
        "criterion": "sensitivity_stable_without_19_68",
        "reference_value": float(ref_sens["balanced_accuracy"]),
        "expanded_value": float(exp_sens["balanced_accuracy"]),
        "delta": sensitivity_ba_delta,
        "pass": sensitivity_ba_delta >= -0.02 and sensitivity_fpr_delta <= 0.05,
    },
    {
        "criterion": "hard_normal_35_48_not_worse",
        "reference_value": int(ref_main["hard_normal_35_48_fp_count"]),
        "expanded_value": int(exp_main["hard_normal_35_48_fp_count"]),
        "delta": hard_normal_fp_delta,
        "pass": hard_normal_fp_delta <= 0,
    },
]
decision_matrix = pd.DataFrame(criteria)
adopt_expanded = bool(decision_matrix["pass"].all())
decision_matrix["final_decision"] = (
    "expanded_compact13_candidate" if adopt_expanded else "candidate_quality_review_required"
)
decision_matrix.to_csv(DECISION_MATRIX_PATH, index=False, encoding="utf-8-sig")

cv_metrics.loc[cv_metrics["fold"].astype(str).eq("all")].sort_values(
    ["evaluation_scope", "strategy"]
)


,strategy,evaluation_scope,fold,n,normal_n,positive_n,balanced_accuracy,precision,recall,f1,false_positive_count,false_negative_count,true_positive_count,true_negative_count,false_positive_rate,hard_normal_35_48_fp_count,review_required_19_68_fp_count
0,dummy_most_frequent,main_all_49,all,49,35,14,0.500000,0.000000,0.000000,0.000000,0,14,0,35,0.000000,0,0
12,expanded_compact13,main_all_49,all,49,35,14,0.850000,0.785714,0.785714,0.785714,3,3,11,32,0.085714,0,1
24,reference_compact13,main_all_49,all,49,35,14,0.828571,0.833333,0.714286,0.769231,2,4,10,33,0.057143,0,1
6,dummy_most_frequent,sensitivity_exclude_review_required_19_68,all,47,33,14,0.500000,0.000000,0.000000,0.000000,0,14,0,33,0.000000,0,0
18,expanded_compact13,sensitivity_exclude_review_required_19_68,all,47,33,14,0.862554,0.846154,0.785714,0.814815,2,3,11,31,0.060606,0,0
30,reference_compact13,sensitivity_exclude_review_required_19_68,all,47,33,14,0.841991,0.909091,0.714286,0.800000,1,4,10,32,0.030303,0,0


## 8. feature importance


In [9]:
def fit_importance_rows(strategy: str, train_rows: pd.DataFrame) -> list[dict]:
    model = make_logistic_pipeline()
    model.fit(train_rows[COMPACT13_FEATURES], train_rows["y"].astype(int))
    coefs = model.named_steps["model"].coef_[0]
    rows = []
    for feature, coef in zip(COMPACT13_FEATURES, coefs):
        rows.append(
            {
                "strategy": strategy,
                "feature": feature,
                "coefficient": float(coef),
                "abs_coefficient": abs(float(coef)),
                "train_rows": int(len(train_rows)),
                "train_positive": int(train_rows["y"].eq(1).sum()),
                "train_normal": int(train_rows["y"].eq(0).sum()),
            }
        )
    result = pd.DataFrame(rows)
    result["importance_rank"] = result["abs_coefficient"].rank(method="first", ascending=False).astype(int)
    return result.to_dict("records")


importance_rows = []
importance_rows.extend(fit_importance_rows("reference_compact13", fixed_eval_model_rows))
importance_rows.extend(
    fit_importance_rows(
        "expanded_compact13",
        pd.concat([fixed_eval_model_rows, candidate_model_rows], ignore_index=True),
    )
)
feature_importance = pd.DataFrame(importance_rows).sort_values(["strategy", "importance_rank"])
feature_importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")
feature_importance.head(10)


,strategy,feature,coefficient,abs_coefficient,train_rows,train_positive,train_normal,importance_rank
14,expanded_compact13,outdoor_temperature__last_6h_mean_minus_prev_6...,1.894215,1.894215,131,26,105,1
13,expanded_compact13,outdoor_temperature__last_12h_mean_minus_prev_...,-1.600682,1.600682,131,26,105,2
19,expanded_compact13,p_net_return_temperature__last_1d_mean_minus_p...,-0.732776,0.732776,131,26,105,3
17,expanded_compact13,p_hc1_return_temperature__last_1d_mean_minus_p...,-0.665760,0.665760,131,26,105,4
24,expanded_compact13,s_hc1_supply_temperature_error__last_minus_first,-0.648145,0.648145,131,26,105,5
16,expanded_compact13,p_hc1_return_temperature__last_12h_mean_minus_...,-0.626526,0.626526,131,26,105,6
22,expanded_compact13,s_hc1_supply_temperature__last_1d_mean_minus_p...,0.431863,0.431863,131,26,105,7
18,expanded_compact13,p_net_meter_flow__last_1d_std_minus_prev_6d_std,-0.304363,0.304363,131,26,105,8
23,expanded_compact13,s_hc1_supply_temperature__last_1d_std_minus_pr...,-0.141768,0.141768,131,26,105,9
20,expanded_compact13,p_return_gap__last_1d_mean_minus_prev_6d_mean,0.109202,0.109202,131,26,105,10


## 9. 보고서 생성


In [10]:
def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    table_df = df[columns].copy()
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in table_df.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, sep] + rows)


summary_table = (
    feature_pool.groupby(["pool_role", "candidate_type", "label", "y"])
    .size()
    .reset_index(name="rows")
)

metric_summary = cv_metrics.loc[cv_metrics["fold"].astype(str).eq("all")].copy()
for col in ["balanced_accuracy", "precision", "recall", "f1", "false_positive_rate"]:
    metric_summary[col] = metric_summary[col].map(lambda value: f"{value:.4f}")

decision_table = decision_matrix.copy()
for col in ["reference_value", "expanded_value", "delta"]:
    decision_table[col] = decision_table[col].map(lambda value: f"{value:.4f}")

accepted_positive_count = int(
    feature_pool.loc[feature_pool["candidate_type"].eq("weak_positive")].shape[0]
)
accepted_normal_count = int(
    feature_pool.loc[feature_pool["candidate_type"].eq("candidate_normal")].shape[0]
)
final_decision_text = (
    "expanded_compact13를 다음 기준 후보로 채택"
    if adopt_expanded
    else "확장 candidate 품질 재검토 필요"
)

report = f"""# M1 Expanded Training Fixed-Eval 검증 보고서

## 개요

이번 단계는 feature/model/threshold를 바꾸지 않고 학습 데이터만 확장했을 때, 기존 `strict_no_event20` 49행 fixed eval 성능이 유지되는지 검증했다.

최종 결론: **{final_decision_text}**

## 무엇을 했는지

- fixed eval 49행은 그대로 평가에만 사용했다.
- weak positive 후보는 coverage 0.95 이상만 학습 후보로 추가했다.
- candidate normal은 disturbance/fault/fixed eval window와 겹치지 않는 7일 window만 선택했다.
- 각 fold에서 test substation과 같은 expansion candidate는 train에서 제외했다.
- Event 19/68은 삭제하지 않고 `review_required_normal` tag로만 유지했다.

## 데이터 구성

{markdown_table(summary_table, ["pool_role", "candidate_type", "label", "y", "rows"])}

## 평가 결과

{markdown_table(metric_summary, ["evaluation_scope", "strategy", "n", "balanced_accuracy", "precision", "recall", "f1", "false_positive_count", "false_negative_count", "false_positive_rate", "hard_normal_35_48_fp_count"])}

## Decision Matrix

{markdown_table(decision_table, ["criterion", "reference_value", "expanded_value", "delta", "pass", "final_decision"])}

## 변경 내용

| 항목 | 내용 |
| --- | --- |
| 노트북 | `06_노트북/13_m1_expanded_training_fixed_eval_validation.ipynb` |
| 후보 audit | `07_데이터산출물/m1_expansion_candidate_audit.csv` |
| feature pool | `07_데이터산출물/m1_expansion_feature_pool.csv` |
| 예측/성능 | `m1_expanded_training_fixed_eval_predictions.csv`, `m1_expanded_training_fixed_eval_cv_metrics.csv` |
| 판단 | `m1_expanded_training_fixed_eval_decision_matrix.csv` |

## 검증

```mermaid
flowchart TD
  A[fixed eval 49 rows] --> B[fold split by substation]
  C[weak positive candidates] --> D[train-only expansion]
  E[candidate normal windows] --> D
  D --> F[test substation candidate exclusion]
  B --> G[fixed eval only evaluation]
  F --> G
  G --> H[decision matrix]
```

- fixed eval은 49행, normal 35행, positive 14행으로 유지됐다.
- 제외 event는 학습/평가 feature pool에 들어가지 않았다.
- Event 19/68은 평가에서 삭제하지 않고 review tag로만 남겼다.
- accepted weak positive coverage는 모두 0.95 이상이다.
- accepted candidate normal은 disturbance/fault/fixed eval window와 겹치지 않는다.
- 모든 fold에서 train/test substation overlap은 0이다.
- 학습 feature는 compact13 13개만 사용했다.

## 한계와 주의점

- expansion candidate는 회사 제공 normal label이 아니라 학습 후보 normal이므로 `candidate_normal`로 분리했다.
- 이번 결과는 모델 저장이나 운영 배포가 아니다.
- fixed eval은 작기 때문에 성능 숫자 하나보다 FP/FN 패턴과 hard normal 악화 여부를 같이 봐야 한다.

## 다음에 볼 것

- expanded 기준이 채택되면, 다음 단계는 candidate normal/weak positive의 수를 늘리는 방향이다.
- 채택되지 않으면, 후보 window 생성 기준과 weak positive 품질 기준을 다시 조정해야 한다.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)


07_데이터산출물\13_M1_expanded_training_fixed_eval_보고서.md


## 10. 검증


In [11]:
assert FEATURE_POOL_PATH.exists()
assert CANDIDATE_AUDIT_PATH.exists()
assert CV_METRICS_PATH.exists()
assert PREDICTIONS_PATH.exists()
assert DECISION_MATRIX_PATH.exists()
assert FEATURE_IMPORTANCE_PATH.exists()
assert REPORT_PATH.exists()

fixed_pool_check = feature_pool.loc[feature_pool["pool_role"].eq("fixed_eval")]
assert len(fixed_pool_check) == 49
assert fixed_pool_check["label"].value_counts().to_dict() == {"normal": 35, "efd_possible": 14}
assert not feature_pool["source_event_id"].dropna().astype(int).isin(EXCLUDED_EVENT_IDS).any()
assert review_required_events == {19, 68}
assert set(
    fixed_pool_check.loc[
        fixed_pool_check["source_event_id"].astype(int).isin([19, 68]), "review_tag"
    ]
) == {"review_required_normal"}

accepted_positive = feature_pool.loc[feature_pool["candidate_type"].eq("weak_positive")]
if not accepted_positive.empty:
    assert accepted_positive["coverage_rate"].ge(0.95).all()

accepted_normal_audit = candidate_audit.loc[
    candidate_audit["candidate_kind"].eq("candidate_normal")
    & candidate_audit["audit_status"].eq("accepted")
]
assert accepted_normal_audit["disturbance_count"].eq(0).all()
assert ~accepted_normal_audit["fault_window_overlap"].fillna(False).astype(bool).any()
assert ~accepted_normal_audit["fixed_eval_window_overlap"].fillna(False).astype(bool).any()

assert fold_audit["group_overlap_count"].eq(0).all()
assert len(COMPACT13_FEATURES) == 13
assert set(decision_matrix["criterion"]) == {
    "balanced_accuracy_not_drop_over_0_02",
    "recall_not_drop_over_0_05",
    "fpr_not_worse_over_0_05",
    "sensitivity_stable_without_19_68",
    "hard_normal_35_48_not_worse",
}

forbidden_terms = ["manufacturer" + "_2", "manufacturer " + "2", "M" + "2"]
check_paths = [
    NOTEBOOK_PATH,
    REPORT_PATH,
    CANDIDATE_AUDIT_PATH,
    FEATURE_POOL_PATH,
    CV_METRICS_PATH,
    PREDICTIONS_PATH,
    DECISION_MATRIX_PATH,
    FEATURE_IMPORTANCE_PATH,
]
hits = []
for path in check_paths:
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    if any(term in text for term in forbidden_terms):
        hits.append(str(path))
assert not hits, "비대상 제조사 문자열 발견"

validation_summary = {
    "fixed_eval_rows": len(fixed_pool_check),
    "accepted_weak_positive": int(len(accepted_positive)),
    "accepted_candidate_normal": int(feature_pool["candidate_type"].eq("candidate_normal").sum()),
    "fold_group_overlap_max": int(fold_audit["group_overlap_count"].max()),
    "compact_feature_count": len(COMPACT13_FEATURES),
    "final_decision": final_decision_text,
}
validation_summary


{'fixed_eval_rows': 49,
 'accepted_weak_positive': 12,
 'accepted_candidate_normal': 70,
 'fold_group_overlap_max': 0,
 'compact_feature_count': 13,
 'final_decision': 'expanded_compact13를 다음 기준 후보로 채택'}